# Lumis 03: Train the classifier

Transfer learning on the frozen split from notebook 02 (CP1 section 3.5.1):

- **Stage 1:** ImageNet backbone frozen, train a new classification head
- **Stage 2:** unfreeze the top of the backbone and fine-tune at a low learning rate
- Class weights handle the imbalance (scabies has about a quarter of eczema's images)
- Evaluated on the **validation set only**. The test set is never loaded here; it is used once, at the very end, on the final chosen model
- The model outputs raw logits, and the validation logits are saved, so notebook 04 can apply temperature scaling

To train a comparator, change `BACKBONE` in step 1 to `'efficientnetb0'` or `'resnet50'` and run all again. Every run is saved under its own name.

**Runtime > Change runtime type > T4 GPU** before running.

## 1. Settings

In [ ]:
BACKBONE = 'efficientnetb0'   # 'mobilenetv2' | 'efficientnetb0' | 'resnet50'
SPLIT = 'v2'                  # v2 = non-skin images removed (notebook 06)
IMG_SIZE = 224
BATCH = 32
SEED = 42
HEAD_EPOCHS = 10       # stage 1, frozen backbone
FINE_EPOCHS = 30       # stage 2, early stopping usually ends it sooner
HEAD_LR = 1e-3
FINE_LR = 5e-5
UNFREEZE_TOP = 0.35    # fraction of backbone layers unfrozen in stage 2

## 2. Setup and data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, json, time
import numpy as np, pandas as pd
import tensorflow as tf
import keras
from keras import layers

BASE = '/content/drive/MyDrive/Lumis'
LOCAL = '/content/dermnet'
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
        z.extractall(LOCAL)

keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE - switch the runtime to T4 GPU')

split = pd.read_csv(f'{BASE}/data/lumis_split_{SPLIT}.csv')
CLASSES = sorted(split.lumis_class.unique())
IDX = {c: i for i, c in enumerate(CLASSES)}
train_df = split[split.split == 'train'].reset_index(drop=True)
val_df = split[split.split == 'val'].reset_index(drop=True)
# The test split is deliberately never loaded in this notebook
print(CLASSES)
print('train', len(train_df), '| val', len(val_df))

## 3. Input pipeline and augmentation

Augmentation follows CP1: flips, rotation, zoom, brightness and contrast. No hue shifts, because they would distort skin tone.

In [ ]:
APPS = {
    'mobilenetv2': (keras.applications.MobileNetV2, keras.applications.mobilenet_v2.preprocess_input),
    'efficientnetb0': (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
    'resnet50': (keras.applications.ResNet50, keras.applications.resnet50.preprocess_input),
}
Backbone, preprocess = APPS[BACKBONE]

def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.uint8), label   # uint8 keeps the in-memory cache small

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
    layers.RandomContrast(0.15),
])

def make_ds(df, training):
    paths = [os.path.join(LOCAL, p) for p in df.path]
    labels = [IDX[c] for c in df.lumis_class]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(df), seed=SEED)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y))
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df, training=False)

from sklearn.utils.class_weight import compute_class_weight
y_train = np.array([IDX[c] for c in train_df.lumis_class])
cw = compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y_train)
CLASS_WEIGHT = dict(enumerate(cw))
print({CLASSES[i]: round(w, 2) for i, w in CLASS_WEIGHT.items()})

## 4. Build the model

The backbone runs in inference mode (`training=False`) so its batch-norm statistics stay fixed during fine-tuning. The last layer outputs logits, not probabilities, so calibration can be applied later.

In [ ]:
base = Backbone(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASSES), name='logits')(x)
model = keras.Model(inputs, outputs)

LOSS = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
RUN = f"{BACKBONE}_{time.strftime('%Y%m%d_%H%M')}"
RUN_DIR = f'{BASE}/exports/runs/{RUN}'
os.makedirs(RUN_DIR, exist_ok=True)
CKPT = f'{BASE}/checkpoints/{RUN}.keras'
print('Run:', RUN)

## 5. Stage 1: train the head

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss=LOSS, metrics=['accuracy'])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=CLASS_WEIGHT,
               callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

## 6. Stage 2: fine-tune the top of the backbone

In [ ]:
base.trainable = True
cut = int(len(base.layers) * (1 - UNFREEZE_TOP))
for layer in base.layers[:cut]:
    layer.trainable = False
# Keep every batch-norm layer frozen. If they train, their statistics shift on the small,
# augmented batches and the pretrained features break (the loss spike seen in run 1)
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
print(f'Fine-tuning {len(base.layers) - cut} of {len(base.layers)} backbone layers')

model.compile(optimizer=keras.optimizers.Adam(FINE_LR), loss=LOSS, metrics=['accuracy'])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_EPOCHS, class_weight=CLASS_WEIGHT,
               callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
                          keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2, min_lr=1e-7),
                          keras.callbacks.ModelCheckpoint(CKPT, save_best_only=True)])
model.save(CKPT)
print('Saved model:', CKPT)

## 7. Training curves

In [ ]:
import matplotlib.pyplot as plt
hist = {k: h1.history[k] + h2.history[k] for k in ['loss', 'val_loss', 'accuracy', 'val_accuracy']}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, m in enumerate(['loss', 'accuracy']):
    ax[i].plot(hist[m], label='train')
    ax[i].plot(hist[f'val_{m}'], label='validation')
    ax[i].axvline(len(h1.history['loss']) - 0.5, color='gray', ls='--', lw=1)
    ax[i].set_title(f'{m} (dashed line = start of fine-tuning)')
    ax[i].set_xlabel('epoch')
    ax[i].legend()
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/training_curves.png', dpi=100)
plt.show()

## 8. Validation metrics

Sensitivity (recall), specificity and F1 per class, as CP1 Table 5 requires. Validation logits are saved for calibration in notebook 04.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

logits = model.predict(val_ds, verbose=0)
y_true = np.array([IDX[c] for c in val_df.lumis_class])
y_pred = logits.argmax(1)
np.save(f'{RUN_DIR}/val_logits.npy', logits)
np.save(f'{RUN_DIR}/val_labels.npy', y_true)

cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
rep = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
rows = []
for i, c in enumerate(CLASSES):
    tp = cm[i, i]; fn = cm[i].sum() - tp; fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
    rows.append({'class': c, 'sensitivity': tp / (tp + fn), 'specificity': tn / (tn + fp),
                 'precision': rep[c]['precision'], 'f1': rep[c]['f1-score'], 'support': int(cm[i].sum())})
metrics = pd.DataFrame(rows).set_index('class')
metrics.loc['macro avg'] = metrics.mean(numeric_only=True)
metrics.loc['macro avg', 'support'] = int(cm.sum())
display(metrics.round(3))
print('Validation accuracy:', round((y_pred == y_true).mean(), 3))

ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap='Blues', xticks_rotation=45)
plt.title(f'{BACKBONE}: validation confusion matrix')
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/confusion_matrix.png', dpi=100)
plt.show()

## 9. Log the run

In [ ]:
metrics.round(4).to_csv(f'{RUN_DIR}/val_metrics.csv')
summary = {
    'run': RUN, 'backbone': BACKBONE, 'split': SPLIT, 'img_size': IMG_SIZE, 'batch': BATCH, 'seed': SEED,
    'head_epochs_run': len(h1.history['loss']), 'fine_epochs_run': len(h2.history['loss']),
    'unfreeze_top': UNFREEZE_TOP, 'head_lr': HEAD_LR, 'fine_lr': FINE_LR,
    'params': int(model.count_params()),
    'val_accuracy': round(float((y_pred == y_true).mean()), 4),
    'val_macro_f1': round(float(metrics.loc['macro avg', 'f1']), 4),
    'val_macro_sensitivity': round(float(metrics.loc['macro avg', 'sensitivity']), 4),
    'val_macro_specificity': round(float(metrics.loc['macro avg', 'specificity']), 4),
    'model_file': CKPT,
}
json.dump(summary, open(f'{RUN_DIR}/summary.json', 'w'), indent=1)

LOG = f'{BASE}/exports/runs_log.csv'
log_df = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
pd.concat([log_df, pd.DataFrame([summary])]).to_csv(LOG, index=False)
print(json.dumps(summary, indent=1))